###Setup library


In [1]:
!pip install -q unsloth trl accelerate bitsandbytes datasets pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.0/353.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 11.6 MB/s eta 0:00:00


In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "microsoft/phi-2",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.3: Fast Phi patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

microsoft/phi-2 does not have a padding token! Will use pad_token = <|endoftext|>.


### Process dataset

In [3]:
import pandas as pd

In [5]:
df = pd.read_csv("/content/test_final.csv")
df = df.dropna(subset=["prompt", "essay", "band"])
df["band"] = df["band"].astype(str).str.strip().replace({"<4": "3.5"}).astype(float)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [7]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an IELTS Writing Task 2 examiner. Evaluate the essay using IELTS band descriptors.
Internally calculate scores for the four criteria, average them, and round to the nearest 0.5.
Return ONLY the overall band score in strict JSON format.

### Input:
Essay prompt: {}
Essay: {}

### Response:
{}"""

### Format to prompt response


In [8]:
import json

EOS_TOKEN = tokenizer.eos_token

def format_band_to_json(band_value):
    return json.dumps({"overallScore": band_value})

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    essays  = examples["essay"]
    outputs = examples["band"]
    texts = []
    for prompt, essay, output in zip(prompts, essays, outputs):
        output_json = format_band_to_json(output)
        text = alpaca_prompt.format(prompt, essay, output_json) + EOS_TOKEN
        texts.append(text)
    return { "text": texts }


In [9]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["prompt", "essay", "band"]])
dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

Map:   0%|          | 0/495 [00:00<?, ? examples/s]

### LoRA



In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 42,
    use_rslora = False,
)


Unsloth: Making `model.base_model.model.model` require gradients


### Config train

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "/content/llm_band_model",
        report_to = "none",
        eval_strategy = "steps",
        eval_steps = 100,
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/445 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

In [12]:
trainer.train()


The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 445 | Num Epochs = 3 | Total steps = 168
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 7,864,320 of 2,787,548,160 (0.28% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.931300,1.999874


Unsloth: Not an error, but PhiForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=168, training_loss=2.071484770093645, metrics={'train_runtime': 928.4256, 'train_samples_per_second': 1.438, 'train_steps_per_second': 0.181, 'total_flos': 1.155602195085312e+16, 'train_loss': 2.071484770093645, 'epoch': 3.0})

In [14]:
model.save_pretrained("/content/llm_band_model")
tokenizer.save_pretrained("/content/llm_band_model")

('/content/llm_band_model/tokenizer_config.json',
 '/content/llm_band_model/special_tokens_map.json',
 '/content/llm_band_model/vocab.json',
 '/content/llm_band_model/merges.txt',
 '/content/llm_band_model/added_tokens.json',
 '/content/llm_band_model/tokenizer.json')

In [15]:
import zipfile, os

def zip_folder(folder_path, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, start=folder_path)
                zipf.write(file_path, arcname)

zip_folder("/content/llm_band_model", "/content/llm_band_model.zip")

### Loss func

In [16]:
from transformers import TextStreamer
import json

streamer = TextStreamer(tokenizer)
predicted_scores = []
true_scores = []

for example in eval_dataset:
    prompt = alpaca_prompt.format(example["prompt"], example["essay"], "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=128)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Trích xuất band từ JSON
    try:
        json_str = decoded.split("### Response:")[-1].strip()
        band = json.loads(json_str)["overallScore"]
        predicted_scores.append(float(band))
        true_scores.append(float(example["band"]))
    except:
        continue  # bỏ qua nếu lỗi định dạng


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score
import numpy as np

mae = mean_absolute_error(true_scores, predicted_scores)
mse = mean_squared_error(true_scores, predicted_scores)
rmse = np.sqrt(mse)

# Ép về string để sklearn coi là nhãn discrete
y_pred_class = y_pred_class.astype(str)
y_true_class = y_true_class.astype(str)

acc = accuracy_score(y_true_class, y_pred_class)
f1 = f1_score(y_true_class, y_pred_class, average='weighted')
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")